In [20]:
from sklearn.datasets import fetch_openml
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA 
from sklearn.preprocessing import StandardScaler

In [1]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', as_frame=False) #as_frame=True would be a df, we want NumpyArray since it has images

The dataset is returned as sklearn.utils.Bunch object. It has DESCR, data (input data) and target (labels) 

In [7]:
mnist.DESCR

"**Author**: Yann LeCun, Corinna Cortes, Christopher J.C. Burges  \n**Source**: [MNIST Website](http://yann.lecun.com/exdb/mnist/) - Date unknown  \n**Please cite**:  \n\nThe MNIST database of handwritten digits with 784 features, raw data available at: http://yann.lecun.com/exdb/mnist/. It can be split in a training set of the first 60,000 examples, and a test set of 10,000 examples  \n\nIt is a subset of a larger set available from NIST. The digits have been size-normalized and centered in a fixed-size image. It is a good database for people who want to try learning techniques and pattern recognition methods on real-world data while spending minimal efforts on preprocessing and formatting. The original black and white (bilevel) images from NIST were size normalized to fit in a 20x20 pixel box while preserving their aspect ratio. The resulting images contain grey levels as a result of the anti-aliasing technique used by the normalization algorithm. the images were centered in a 28x28 

In [9]:
X, y = mnist.data, mnist.target
X.shape

(70000, 784)

70000 images , each image has 784 features (28 x 28 pixels). One feature = one pixels intensity.

In [11]:
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]

In [ ]:
f

In [32]:
from sklearn.model_selection import GridSearchCV 
from sklearn.preprocessing import MinMaxScaler #works better for pictures

knn_classifier_pipeline = make_pipeline(
    MinMaxScaler(), # PCA and KNN work best on scaled data # PCA: reduces dimensions! 
    PCA(n_components=0.95, random_state=42), # we keep enough dimensions to explain 95% of the variance 
    KNeighborsClassifier()
)

# in a knn-model the hyperparameters: weights and n_neighbours - can have different setting
# with gridsearch we try all combinations of weights and neighbors
param_grid = [ # when using a pipeline in GridSearchCv we must say which part of the pipeline the hyperparametres belong to with__
    {'kneighborsclassifier__weights': ["uniform", "distance"], #-uniform: all neighbors have equal vote, -distance: close neighbors have more weight
     'kneighborsclassifier__n_neighbors': list(range(3,8))} #amount of neighbours the modell will look at to determine class
]  # 2,3,4

#configurate gridsearch
grid_search = GridSearchCV(knn_classifier_pipeline, param_grid, cv=5, n_jobs=-1, verbose=3)
# train the gridsearch
grid_search.fit(X_train, y_train)



Fitting 5 folds for each of 10 candidates, totalling 50 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'kneighborsclassifier__n_neighbors': [3, 4, ...], 'kneighborsclassifier__weights': ['uniform', 'distance']}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is

In [35]:
grid_search.best_params_

{'kneighborsclassifier__n_neighbors': 4,
 'kneighborsclassifier__weights': 'distance'}

In [36]:
from sklearn.metrics import accuracy_score

final_model = grid_search.best_estimator_ # this is the retrained (automatical) model with the best parameters found by GridSearchCV

y_pred = final_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
accuracy 

0.9731

Huh. I first got 95%. Then I changed to MinMaxScaler and got 97%